# Second model pilot: pi0.5 on LIBERO-Spatial (about 1-2 h, GPU T4 x2)
Checks that pi0.5 runs on Kaggle before we spend quota on the full experiment. **Run interactively** (Shift+Enter
cell by cell), not as a commit, so we can stop early if something fails.

**Setup:** Settings -> Accelerator **GPU T4 x2**, Internet **On**. Add-ons -> Secrets -> `HF_TOKEN` (a Hugging Face
read token whose account has accepted google/paligemma-3b-pt-224), switched **on** for this notebook.
Add Input: dataset **vla-assure** (latest version). **Send back:** `pi05_pilot.zip` (Output / file browser) or
a screenshot of the last cell.

## 1. Setup

In [ ]:
# 1a. GPUs, time guard, work folder; resume from a previous version's output if one is attached as input.
import os, sys, time, json, glob, shutil, pathlib, subprocess
T_START = time.time()
TIME_BUDGET_H = 11.0          # stop cleanly before Kaggle's 12 h limit so the output is saved
def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU >= 1, "No GPU: right panel > Accelerator > GPU T4 x2"
print("GPUs:", NGPU, "| kernel Python", sys.version.split()[0])

WORK = pathlib.Path("/kaggle/working/vla-gonogo"); WORK.mkdir(parents=True, exist_ok=True)
# Kaggle mounts inputs at different depths (e.g. /kaggle/input/notebooks/<user>/<slug>/vla-gonogo)
prev = sorted({p for p in glob.glob("/kaggle/input/**/vla-gonogo", recursive=True) if pathlib.Path(p).is_dir()})
for p in prev:
    if pathlib.Path(p).resolve() != WORK.resolve():
        print("Resuming from previous output:", p)
        shutil.copytree(p, WORK, dirs_exist_ok=True, symlinks=False)
n_done = len(list(WORK.rglob("*.npz")))
print("Work folder:", WORK, "|", n_done, "episodes already on disk")
PERSISTENT = True   # /kaggle/working is saved as the version's output

In [ ]:
# 1b. Code: the newest vla-assure copy among the inputs (needs the dtype switch added for pi0.5)
import zipfile
CODE = pathlib.Path("/root/vla-assure")
def newest(root):
    root = pathlib.Path(root)
    a = root / "src/assure/adapters/lerobot_adapter.py"
    return a.exists() and "pcfg.dtype" in a.read_text() and (root / "scripts/resplit.py").exists()
if CODE.exists() and not newest(CODE):
    shutil.rmtree(CODE)
if not CODE.exists():
    trees = sorted({str(pathlib.Path(p).parents[3]) for p in
                    glob.glob("/kaggle/input/**/src/assure/adapters/lerobot_adapter.py", recursive=True)})
    trees = [t for t in trees if newest(t)]
    if not trees:
        raise FileNotFoundError("No vla-assure copy with the pi0.5 dtype switch: upload the newest vla-assure.zip "
                                "as a new dataset version and attach it")
    shutil.copytree(trees[0], CODE, symlinks=False)
for d in ("rollouts", "results"):
    (WORK / d).mkdir(exist_ok=True)
    tgt = CODE / d
    if tgt.is_symlink(): tgt.unlink()
    elif tgt.exists(): shutil.rmtree(tgt)
    tgt.symlink_to(WORK / d)
os.chdir(CODE); print("Code OK at", os.getcwd())

In [ ]:
# 1b'. Hugging Face token (pi0.5 uses Google's gated PaliGemma tokenizer)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN loaded (length", len(os.environ["HF_TOKEN"]), ")")
except Exception as e:
    raise RuntimeError("Add-ons -> Secrets: add HF_TOKEN and switch it on for this notebook") from e

In [ ]:
# 1c. Python 3.12 venv for LeRobot 0.6.1 (needs Python >= 3.12; Kaggle's kernel env is left untouched).
#     egl_probe / hf-egl-probe (via hf-libero -> robomimic) are C++ packages that fail to build with CMake 4;
#     LeRobot's LIBERO env never imports them, so tiny pure-Python stand-ins are installed first.
VENV = pathlib.Path("/root/py312"); PY = str(VENV / "bin/python")
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       env=dict(os.environ, CMAKE_POLICY_VERSION_MINIMUM="3.5"))
    if r.returncode != 0 or not check:
        print(r.stdout[-1500:], r.stderr[-3000:])
    if check:
        assert r.returncode == 0, f"failed: {cmd}"
    return r
if not pathlib.Path(PY).exists():
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --seed --python 3.12 {VENV}")
stub = pathlib.Path("/root/_stubs")
(stub / "egl_probe/egl_probe").mkdir(parents=True, exist_ok=True); (stub / "hf_egl_probe").mkdir(parents=True, exist_ok=True)
(stub / "egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "egl_probe"\nversion = "1.0.2"\n[tool.setuptools]\npackages = ["egl_probe"]\n')
(stub / "egl_probe/egl_probe/__init__.py").write_text("def get_available_devices():\n    return [0]\n")
(stub / "hf_egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "hf-egl-probe"\nversion = "1.0.2"\n[tool.setuptools]\npy-modules = []\n')
sh(f"{PY} -m pip install -q {stub / 'egl_probe'} {stub / 'hf_egl_probe'}")
t = time.time()
sh(f'{PY} -m pip install -q "lerobot[pi,smolvla,libero]==0.6.1" pandas scipy tabulate pyyaml')
print(f"installed in {time.time() - t:.0f} s:",
      sh(f"{PY} -c \"import importlib.metadata as m, torch; print('lerobot', m.version('lerobot'), '| torch', "
         f"torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count())\"").stdout)

In [ ]:
# 1c'. Can this token read the gated tokenizer? (fails fast with a clear message instead of deep inside LeRobot)
r = subprocess.run([PY, "-c", "from transformers import AutoTokenizer; "
                    "AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224'); print('tokenizer OK')"],
                   capture_output=True, text=True, env=dict(os.environ))
print((r.stdout + r.stderr)[-800:])
assert r.returncode == 0, ("token cannot read google/paligemma-3b-pt-224: open that page on huggingface.co with the "
                           "same account, click 'Agree and access', then rerun this cell")

In [ ]:
# 1d. Headless rendering. Try NVIDIA EGL first; if MuJoCo cannot render, install Mesa EGL (software) and retry;
#     last resort OSMesa. The chosen backend is used by every later step.
probe = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos=\"0 0 1\"/>"
         "<geom type=\"sphere\" size=\".1\"/></worldbody></mujoco>'); r = mujoco.Renderer(m, 64, 64); "
         "d = mujoco.MjData(m); mujoco.mj_forward(m, d); r.update_scene(d); print('render ok', r.render().shape)")
def try_gl(gl):
    r = subprocess.run([PY, "-c", probe], capture_output=True, text=True,
                       env=dict(os.environ, MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl))
    return r.returncode == 0, (r.stdout + r.stderr)[-400:]
GL = None
for attempt in ("egl", "egl+mesa", "osmesa"):
    if attempt == "egl+mesa":
        sh("apt-get -qq update && apt-get -qq install -y libegl1 libegl-mesa0 libgl1 libgl1-mesa-dri >/dev/null", check=False)
    if attempt == "osmesa":
        sh("apt-get -qq install -y libosmesa6 >/dev/null", check=False)
    ok, log = try_gl(attempt.split("+")[0])
    print(attempt, "->", "OK" if ok else log)
    if ok:
        GL = attempt.split("+")[0]; break
assert GL, "no rendering backend works"
if attempt == "egl+mesa":
    print("NOTE: software EGL (Mesa). Rendering is CPU-bound; expect slower episodes than on Colab.")
os.environ["MUJOCO_GL"] = GL; os.environ["PYOPENGL_PLATFORM"] = GL
os.environ["MPLBACKEND"] = "Agg"   # the kernel's inline backend is not valid inside the venv's matplotlib
# LIBERO's first import asks interactive questions and downloads assets: answer once, non-interactively.
r = subprocess.run([PY, "-c", "import libero.libero as L; print('assets:', L.get_libero_path('assets'))"],
                   input="n\n", capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-600:], r.stderr[-600:])

In [ ]:
# 1e. Experiment configuration for this gate (written into the repo copy on this machine only)
import yaml, json
CFG = CODE / "configs"
exp = yaml.safe_load((CFG / "experiment.yaml").read_text())
models = yaml.safe_load((CFG / "models.yaml").read_text())

MODEL, SUITE = "pi05", "libero_spatial"
CONDITION, KEY = "nominal", "libero_spatial"
N_ACTION_STEPS = 10     # same throughput setting as SmolVLA
DTYPE = "float32"       # set by the dtype test in 2a ("float32" or "bfloat16")
X = "model.paligemma_with_expert."
exp["quant_scope"] = {"include": [X + "gemma_expert.*"], "block_regex": r"gemma_expert\.model\.layers\.(\d+)$"}
exp["variants"]["w4_lm"] = {"kind": "quant", "bits": 4, "group_size": 128, "own_scope": True,
                            "include": [X + "paligemma.model.language_model.*", X + "gemma_expert.*"]}
exp["variants"]["w8_lm"] = dict(exp["variants"]["w4_lm"], bits=8)
models["models"][MODEL]["n_action_steps"] = N_ACTION_STEPS
def write_cfg(dtype):
    models["models"][MODEL]["dtype"] = dtype
    (CFG / "experiment.yaml").write_text(yaml.safe_dump(exp, sort_keys=False))
    (CFG / "models.yaml").write_text(yaml.safe_dump(models, sort_keys=False))
write_cfg(DTYPE)
print("checkpoint:", models["models"][MODEL]["checkpoint"], "| update:", models["models"][MODEL].get("update_checkpoint"))
print("taps:", models["models"][MODEL]["tap"])

def run(cmd, tail=3000):
    # Run a pipeline script, stream nothing, print the end of the log, raise on failure.
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    p = subprocess.run([PY] + cmd, capture_output=True, text=True, env=env, input="n\n")
    out = (p.stdout + p.stderr).replace("\r", "\n")
    out = "\n".join(l for l in out.splitlines() if "Running rollout" not in l and "it/s" not in l)
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


class OutOfTime(Exception):
    pass

def _clean_partial(since):
    # remove episode files that a killed shard may have left half-written
    import numpy as np
    for f in (CODE / "rollouts").rglob("*.npz"):
        if f.stat().st_mtime >= since - 120:
            try:
                np.load(f)["success"]
            except Exception:
                print("removing partial file", f); f.unlink()

def collect(args, tail=1500):
    # 01_collect.py sharded by task over the GPUs, then one merge pass (skips finished episodes) that writes the
    # complete run.json. Stops cleanly when TIME_BUDGET_H is reached; rerun (next version) to resume.
    if hours_left() < 0.25:
        raise OutOfTime("time budget reached: Save Version now; attach this output and run again to resume")
    args = list(args) + ([] if "--skip_existing" in args else ["--skip_existing"])
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    if NGPU > 1 and "--tasks" not in args:
        logs = [open(f"/root/shard{g}.log", "w") for g in range(NGPU)]
        t0 = time.time()
        procs = [subprocess.Popen([PY, "scripts/01_collect.py"] + args +
                                  ["--tasks", ",".join(str(t) for t in range(10) if t % NGPU == g)],
                                  # all shards on GPU 0 (the setup that works with robosuite's EGL checks); SmolVLA is small and the
                                  # simulator is CPU-bound, so two processes on one T4 still give ~2x
                                  env=dict(env, CUDA_VISIBLE_DEVICES="0"), stdout=logs[g], stderr=subprocess.STDOUT,
                                  stdin=subprocess.PIPE, text=True) for g in range(NGPU)]
        for p in procs:
            p.stdin.write("n\n"); p.stdin.close()
        # poll: stop all shards at once if one fails or the time budget runs out
        bad = []
        while any(p.poll() is None for p in procs):
            bad = [g for g, p in enumerate(procs) if p.poll() not in (None, 0)]
            late = hours_left() * 3600 < 600
            if bad or late:
                for p in procs:
                    if p.poll() is None:
                        p.kill()
                for p in procs:
                    p.wait()
                _clean_partial(t0)
                if late and not bad:
                    raise OutOfTime("time budget reached during collection: Save Version; attach this output and run again")
                break
            time.sleep(10)
        for f in logs:
            f.close()
        for g, p in sorted(enumerate(procs), key=lambda x: x[0] not in bad):   # the shard that failed first
            if p.returncode != 0:
                print(open(f"/root/shard{g}.log").read()[-3000:])
                raise RuntimeError(f"shard {g} failed")
    run(["scripts/01_collect.py"] + args, tail=tail)   # merge pass (or the single-GPU run)

## 2. Pilot

In [ ]:
# 2a. Precision test: one episode of task 0 in float32 and in bfloat16 (T4 has no native bfloat16, so it may be slow)
import numpy as np
P = {}
def trial(tag, extra, env_extra=None, tail=1200):
    out = WORK / f"pilot/{tag}"
    cmd = [PY, "scripts/01_collect.py", "--model", MODEL, "--suite", SUITE, "--condition", "nominal", "--out", str(out)] + extra
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false", CUDA_VISIBLE_DEVICES="0",
               **(env_extra or {}))
    t = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, env=env, input="n\n")
    log = "\n".join(l for l in (p.stdout + p.stderr).replace("\r", "\n").splitlines() if "it/s" not in l)
    rj = next(iter(out.rglob("run.json")), None)
    rj = json.loads(rj.read_text()) if rj else {}
    res = dict(ok=p.returncode == 0, wall_s=round(time.time() - t), s_per_ep=round(rj.get("seconds", 0) / max(rj.get("episodes", 1), 1)),
               success=rj.get("success_rate"), episodes=rj.get("episodes"), peak_gib=rj.get("peak_gpu_mem_gib"),
               taps=rj.get("tap_fired"), query_s=rj.get("query_latency_median_s"))
    P[tag] = res
    print(f"[{tag}] {res}")
    if not res["ok"]:
        print(log[-tail:])
    return res

for dt in ("float32", "bfloat16"):
    write_cfg(dt)
    trial(f"dtype_{dt}", ["--variant", "fp16", "--tasks", "0", "--episodes", "1"])
ok = {dt: P[f"dtype_{dt}"] for dt in ("float32", "bfloat16") if P[f"dtype_{dt}"]["ok"]}
assert ok, "pi0.5 did not run in either precision: send me the log above"
DTYPE = min(ok, key=lambda d: ok[d]["s_per_ep"])
write_cfg(DTYPE)
print("\nUsing", DTYPE, "| taps fired:", ok[DTYPE]["taps"])
assert all(v > 0 for v in (ok[DTYPE]["taps"] or {"none": 0}).values()), "feature taps did not fire: send me this output"

In [ ]:
# 2b. Speed and success: one episode per task, all 10 tasks (~10-30 min)
r = trial("fp16_10tasks", ["--variant", "fp16", "--episodes", "1"])
print(f"\nfp16 success {r['success']} over {r['episodes']} episodes | {r['s_per_ep']} s/episode | peak {r['peak_gib']} GiB")

In [ ]:
# 2c. Model changes load and run: W4 / W8 on the action expert, W4 on language model + expert, the second checkpoint
for v in ("w4", "w8", "w4_lm", "update"):
    trial(f"{v}", ["--variant", v, "--tasks", "0,1,2", "--episodes", "40", "--roles", "test", "--max_per_task", "1"])

In [ ]:
# 2d. Can a second process use GPU 1? (doubles throughput if yes; rendering stays on GPU 0)
trial("gpu1", ["--variant", "fp16", "--tasks", "1", "--episodes", "1", "--device", "cuda:1"],
      env_extra={"CUDA_VISIBLE_DEVICES": "0,1"})

In [ ]:
# 2e. Shadow copy memory: fp16 robot + a W4 shadow on the same GPU (needed for zero-rollout shadow replay)
trial("shadow_w4", ["--variant", "fp16", "--shadow", "w4", "--tasks", "0", "--episodes", "1"])

In [ ]:
# 2f. Summary and size of the full run
import pandas as pd
df = pd.DataFrame(P).T[["ok", "s_per_ep", "success", "episodes", "peak_gib", "query_s"]]
print(df.to_string())
s = P["fp16_10tasks"]["s_per_ep"] or P[f"dtype_{DTYPE}"]["s_per_ep"]
par = 2 if P.get("gpu1", {}).get("ok") else 1
for name, n in (("reference 400", 400), ("one variant (140 test + 80 repair)", 220)):
    print(f"{name:36s} ~ {n * s / 3600 / par:5.1f} GPU-session hours ({par} process(es) in parallel)")
json.dump({"dtype": DTYPE, "pilot": P}, open(WORK / "pi05_pilot.json", "w"), indent=1, default=str)
with zipfile.ZipFile("/kaggle/working/pi05_pilot.zip", "w") as z:
    z.write(WORK / "pi05_pilot.json", "pi05_pilot.json")
    for f in (WORK / "pilot").rglob("run.json"):
        z.write(f, str(f.relative_to(WORK)))
print("Send me /kaggle/working/pi05_pilot.zip or a screenshot of this table")